# 遗传算法（Genetic Algorithm, GA）

遗传算法是一种模拟生物进化过程的智能优化算法。它借鉴了自然选择、遗传、交叉、变异等思想，用一组候选解不断进化，逐步逼近最优解。

遗传算法适合求解：

- 目标函数复杂、不可导的问题；
- 非线性、多峰、非凸优化问题；
- 离散优化和组合优化问题；
- 参数寻优、路径规划、排产调度、特征选择等问题。

一句话理解：遗传算法就是让一群候选解像生物种群一样不断“繁殖、竞争、变异”，让优秀解更容易保留下来。

## 1. 基本概念

遗传算法中常用的术语如下：

| 遗传算法术语 | 优化问题含义 |
| --- | --- |
| 个体 | 一个候选解 |
| 种群 | 一组候选解 |
| 染色体 | 候选解的编码形式 |
| 基因 | 染色体中的一个变量或片段 |
| 适应度 | 衡量候选解好坏的指标 |
| 选择 | 保留较优秀的个体 |
| 交叉 | 两个个体交换部分信息生成新个体 |
| 变异 | 随机改变个体中的部分基因 |

在数学建模里，可以把一个个体看成一个决策变量向量：

$$
x = (x_1, x_2, \dots, x_d)
$$

目标是找到使目标函数最好的一组变量。

## 2. 优化问题形式

以最小化问题为例：

$$
\min_{x \in \Omega} f(x)
$$

其中：

- $x$ 是决策变量；
- $f(x)$ 是目标函数；
- $\Omega$ 是变量的可行域。

遗传算法本身更习惯处理“适应度越大越好”的问题。如果原问题是最小化，可以把目标函数值转化为适应度，例如：

$$
fitness(x) = \frac{1}{1 + f(x)}
$$

不过在代码实现中，也可以直接保留目标函数值，并用较小的目标函数值表示更优秀。

## 3. 编码方式

遗传算法首先要决定如何表示一个解，这叫编码。

常见编码方式：

- 二进制编码：适合 0-1 选择、整数近似、传统 GA 教学；
- 实数编码：适合连续变量优化；
- 排列编码：适合旅行商问题、排序问题、路径规划；
- 混合编码：适合变量类型较复杂的问题。

本笔记主要使用实数编码，因为它在数学建模连续优化问题中最常见。

## 4. 核心操作

遗传算法主要包括四个操作。

### 4.1 初始化

在变量范围内随机生成一批个体，形成初始种群。

### 4.2 选择

选择是让优秀个体更容易进入下一代。常见选择方法有：

- 轮盘赌选择：适应度越高，被选中概率越大；
- 锦标赛选择：随机抽取若干个体，选其中最优秀者；
- 精英保留：直接保留当前最好的若干个体。

### 4.3 交叉

交叉是让两个父代个体交换信息，产生子代。实数编码中常用算术交叉：

$$
child_1 = \alpha parent_1 + (1 - \alpha) parent_2
$$

$$
child_2 = (1 - \alpha) parent_1 + \alpha parent_2
$$

其中 $\alpha \in [0,1]$ 是随机数。

### 4.4 变异

变异是对个体的部分基因进行随机扰动，用来保持种群多样性，避免算法过早陷入局部最优。

## 5. 算法流程

遗传算法的一般流程如下：

1. 设置种群规模、最大迭代次数、交叉概率、变异概率等参数。
2. 随机生成初始种群。
3. 计算每个个体的目标函数值或适应度。
4. 根据适应度进行选择。
5. 对选出的父代进行交叉，产生子代。
6. 对子代进行变异。
7. 修复越界或不满足约束的个体。
8. 形成新一代种群。
9. 重复迭代，直到达到停止条件。
10. 输出最优个体和最优目标函数值。

In [ ]:
import numpy as np

try:
    import matplotlib.pyplot as plt
    plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS"]
    plt.rcParams["axes.unicode_minus"] = False
except ModuleNotFoundError:
    plt = None
    print("当前环境没有安装 matplotlib，绘图单元会跳过。可运行：pip install matplotlib")


## 6. 从零实现实数编码遗传算法

下面实现一个通用的最小化版本遗传算法。为了便于理解，代码采用：

- 实数编码；
- 锦标赛选择；
- 算术交叉；
- 高斯变异；
- 精英保留。

In [ ]:
def genetic_algorithm_minimize(
    func,
    bounds,
    population_size=80,
    generations=200,
    crossover_rate=0.85,
    mutation_rate=0.08,
    mutation_scale=0.1,
    elite_size=2,
    tournament_size=3,
    seed=42,
):
    """使用实数编码遗传算法最小化目标函数。

    Parameters
    ----------
    func : callable
        目标函数，输入形如 (population_size, dim) 的二维数组，返回每个个体的函数值。
    bounds : list[tuple[float, float]]
        每个变量的上下界，例如 [(-5, 5), (-5, 5)]。
    population_size : int
        种群规模。
    generations : int
        进化代数。
    crossover_rate : float
        交叉概率。
    mutation_rate : float
        单个基因发生变异的概率。
    mutation_scale : float
        变异扰动强度，占变量范围的比例。
    elite_size : int
        每代直接保留的优秀个体数量。
    tournament_size : int
        锦标赛选择中每次比较的个体数量。
    seed : int
        随机种子，便于复现实验。
    """
    rng = np.random.default_rng(seed)
    bounds = np.asarray(bounds, dtype=float)
    lower = bounds[:, 0]
    upper = bounds[:, 1]
    span = upper - lower
    dim = len(bounds)

    population = rng.uniform(lower, upper, size=(population_size, dim))
    values = func(population)
    history = []

    def tournament_select(values):
        candidate_indices = rng.integers(0, population_size, size=tournament_size)
        best_local_index = candidate_indices[np.argmin(values[candidate_indices])]
        return population[best_local_index].copy()

    for _ in range(generations):
        order = np.argsort(values)
        elites = population[order[:elite_size]].copy()
        best_value = values[order[0]]
        history.append(best_value)

        new_population = [elite for elite in elites]

        while len(new_population) < population_size:
            parent1 = tournament_select(values)
            parent2 = tournament_select(values)

            if rng.random() < crossover_rate:
                alpha = rng.random(size=dim)
                child1 = alpha * parent1 + (1 - alpha) * parent2
                child2 = (1 - alpha) * parent1 + alpha * parent2
            else:
                child1 = parent1.copy()
                child2 = parent2.copy()

            for child in (child1, child2):
                mutation_mask = rng.random(size=dim) < mutation_rate
                noise = rng.normal(0, mutation_scale * span, size=dim)
                child[mutation_mask] += noise[mutation_mask]
                child = np.clip(child, lower, upper)
                new_population.append(child)
                if len(new_population) >= population_size:
                    break

        population = np.array(new_population)
        values = func(population)

    best_index = np.argmin(values)
    best_x = population[best_index]
    best_y = values[best_index]
    history.append(best_y)

    return best_x, best_y, np.array(history)


## 7. 示例一：Sphere 函数

Sphere 函数是连续优化算法的基础测试函数：

$$
f(x) = \sum_{j=1}^{d} x_j^2
$$

全局最优解为 $x=(0,0,\dots,0)$，最优值为 $0$。

In [ ]:
def sphere(x):
    return np.sum(x ** 2, axis=1)


best_x, best_y, history = genetic_algorithm_minimize(
    sphere,
    bounds=[(-5, 5), (-5, 5)],
    population_size=80,
    generations=150,
    seed=1,
)

print("最优位置：", best_x)
print("最优函数值：", best_y)


In [ ]:
if plt is None:
    print("跳过绘图：请先安装 matplotlib。")
else:
    plt.figure(figsize=(7, 4))
    plt.plot(history, linewidth=2)
    plt.xlabel("进化代数")
    plt.ylabel("当前最优目标函数值")
    plt.title("遗传算法在 Sphere 函数上的收敛曲线")
    plt.grid(alpha=0.3)
    plt.show()


## 8. 示例二：Rastrigin 函数

Rastrigin 函数是一个典型的多峰函数：

$$
f(x) = 10d + \sum_{j=1}^{d}\left[x_j^2 - 10\cos(2\pi x_j)\right]
$$

这个函数有大量局部最优点，更能体现遗传算法通过交叉和变异保持全局搜索能力的特点。

In [ ]:
def rastrigin(x):
    dim = x.shape[1]
    return 10 * dim + np.sum(x ** 2 - 10 * np.cos(2 * np.pi * x), axis=1)


best_x, best_y, history = genetic_algorithm_minimize(
    rastrigin,
    bounds=[(-5.12, 5.12), (-5.12, 5.12)],
    population_size=120,
    generations=300,
    mutation_rate=0.10,
    mutation_scale=0.08,
    seed=2,
)

print("最优位置：", best_x)
print("最优函数值：", best_y)


In [ ]:
if plt is None:
    print("跳过绘图：请先安装 matplotlib。")
else:
    plt.figure(figsize=(7, 4))
    plt.semilogy(history + 1e-12, linewidth=2)
    plt.xlabel("进化代数")
    plt.ylabel("当前最优目标函数值（对数坐标）")
    plt.title("遗传算法在 Rastrigin 函数上的收敛曲线")
    plt.grid(alpha=0.3)
    plt.show()


## 9. 参数解释与调参建议

| 参数 | 常见取值 | 作用 | 调参建议 |
| --- | --- | --- | --- |
| 种群规模 | 30 到 200 | 控制搜索多样性 | 复杂问题适当增大 |
| 进化代数 | 100 到 1000 | 控制搜索时间 | 收敛慢就增加代数 |
| 交叉概率 | 0.6 到 0.95 | 控制信息重组频率 | 一般取较大值 |
| 变异概率 | 0.01 到 0.15 | 控制随机探索能力 | 早熟收敛时增大 |
| 变异强度 | 变量范围的 1% 到 20% | 控制扰动幅度 | 后期可逐渐减小 |
| 精英数量 | 1 到 5 | 保证优秀解不丢失 | 不宜过大，否则多样性下降 |

一般经验：

- 对连续优化问题，可先使用实数编码；
- 对组合优化问题，要根据问题设计特殊编码和交叉方式；
- 如果算法很快停在不理想结果，通常说明多样性不足，可以增大种群规模或变异概率；
- 如果算法波动大、难以稳定收敛，可以降低变异强度或增加精英保留。

## 10. 约束问题如何处理

很多数学建模问题都有约束条件。遗传算法处理约束的常见方法有：

- 边界裁剪：变量超出上下界时直接拉回边界内；
- 罚函数法：违反约束时给目标函数加惩罚项；
- 修复法：把不可行解修正成可行解；
- 可行性优先规则：可行解优先于不可行解，不可行解之间比较违反程度。

罚函数法是建模中最常用的写法。例如原问题为：

$$
\min f(x), \quad g(x) \le 0
$$

可以构造新的目标函数：

$$
F(x) = f(x) + M \max(0, g(x))^2
$$

其中 $M$ 是足够大的罚因子。

## 11. 与粒子群优化算法的比较

| 方面 | 遗传算法 GA | 粒子群优化 PSO |
| --- | --- | --- |
| 灵感来源 | 生物进化 | 群体觅食 |
| 主要操作 | 选择、交叉、变异 | 速度更新、位置更新 |
| 编码灵活性 | 很强，适合离散和组合问题 | 更自然适合连续问题 |
| 参数数量 | 相对较多 | 相对较少 |
| 多样性维护 | 依靠交叉和变异 | 依靠群体分布和随机项 |
| 收敛特点 | 通常较稳但计算量较大 | 通常较快但可能早熟 |

简单选择建议：

- 连续参数优化：GA 和 PSO 都可以，PSO 通常更简洁；
- 排序、选择、路径、调度等组合优化：GA 往往更方便设计；
- 不知道目标函数结构、只需要黑箱寻优：GA 是很通用的备选方法。

## 12. 数学建模中的写作模板

在论文或报告中介绍遗传算法，可以按下面结构写：

1. 决策变量编码：说明染色体如何表示一个解。
2. 适应度函数设计：说明目标函数或罚函数如何计算。
3. 初始种群生成：说明变量范围和随机初始化方式。
4. 遗传操作设计：说明选择、交叉、变异方法。
5. 参数设置：给出种群规模、迭代次数、交叉概率、变异概率。
6. 终止条件：如达到最大迭代次数或最优值变化很小。
7. 结果分析：报告最优解、目标函数值、收敛曲线和敏感性分析。

## 13. 小结

遗传算法的优点：

- 不要求目标函数连续或可导；
- 全局搜索能力较强；
- 编码方式灵活，能处理连续、离散、组合问题；
- 容易与其他算法结合。

遗传算法的缺点：

- 参数较多，需要调参；
- 计算量可能较大；
- 若交叉、变异设计不合理，效果会明显下降；
- 不能保证每次都找到理论全局最优解。

一句话记忆：遗传算法不是直接沿着梯度下降，而是让一批候选解通过选择、交叉、变异不断进化，逐渐筛出更好的解。